In [1]:
from pathlib import Path
import json
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display

## Sample100

In [2]:
dir_sample100 = Path('/projects/mtg/projects/sample-identification/datasets/sample100_withnoise')

path_csv_sample100_samples = dir_sample100 / 'meta' / 'samples.csv'
path_csv_sample100_tracks = dir_sample100 / 'meta' / 'tracks.csv'
path_csv_sample100_yt = dir_sample100 / 'meta' / 'youtube-links.csv'

In [3]:
df_sample100_samples = pd.read_csv(path_csv_sample100_samples, delimiter=',', encoding='latin-1')

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_sample100_samples)

,sample_id,original_track_id,sample_track_id,t_original,t_sample,n_repetitions,sample_type,interpolation,comments
0,S001,T002,T001,0,0,19,riff,no,starts counted (sample ABCD looped ABCBCBCD)
1,S002,T003,T004,0,0,4,riff,no,intro
2,S003,T003,T004,34,53,3,riff,no,refrain with horns
3,S004,T005,T006,0,37,60,beat,no,
4,S005,T007,T008,53,0,10,riff,no,(very clean: same pitch and only layer)
5,S006,T010,T009,0,1,37,beat,no,probably samples both bars (with two decks?)
6,S007,T011,T012,1,0,85,riff,no,(sample a bit gated in the end)
7,S008,T014,T013,74,5,10,riff,maybe,beginning of riff counted (rest 'chopped')
8,S009,T015,T016,0,3,37,riff,yes,interpolation or other version
9,S010,T018,T017,0,0,1,riff,no,


In [4]:
sample100_dict_gt = {}
for _,row in df_sample100_samples.iterrows():
    sample100_dict_gt[row["sample_id"]] = {
        "src_audio_id": row["original_track_id"],
        "dst_audio_id": row["sample_track_id"],
    }

In [5]:
with open("sample100-ground-truth.json", "w") as out_f:
    json.dump(sample100_dict_gt, out_f)

In [4]:
df_sample100_tracks = pd.read_csv(path_csv_sample100_tracks, delimiter=',', skipinitialspace=True)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_sample100_tracks)

,track_id,artist,title,year,genre
0,T001,OC,Time's Up,1994,Hip-hop
1,T002,Les Demerle,A Day in the Life,1968,Jazz
2,T003,David Axelrod,Holy Thursday,1968,Jazz
3,T004,Lil Wayne,Dr. Carter,2008,Hip-hop
4,T005,Clyde McPhatter,Mixed Up Cup,1970,R&B/Soul
5,T006,Common,In My Own World (Check the Method),1994,Hip-hop
6,T007,Monty Alexander,Love and Happiness,1974,Jazz
7,T008,The Beatnuts,Let Off a Couple,1994,Hip-hop
8,T009,Beastie Boys,Rhymin & Stealin',1986,Hip-hop
9,T010,Led Zeppelin,When The Levee Breaks,1971,Rock


In [5]:
# This is not the official file
df_sample100_yt = pd.read_csv(path_csv_sample100_yt, delimiter=',', skipinitialspace=True, header=None)
display(df_sample100_yt)

,0,1
0,T001,https://www.youtube.com/watch?v=dcsPoM2MalY
1,T002,https://www.youtube.com/watch?v=DkQFAxSwwjk
2,T003,https://www.youtube.com/watch?v=8j8pSu3U7WM
3,T004,https://www.youtube.com/watch?v=es6goNuH0lY
4,T005,https://www.youtube.com/watch?v=3ofnKqgTdSk
...,...,...
139,T190,https://www.youtube.com/watch?v=sulaqzO0ICk
140,T191,https://www.youtube.com/watch?v=vimZj8HW0Kg
141,T192,https://www.youtube.com/watch?v=e4Zey7o04Qk
142,T193,https://www.youtube.com/watch?v=E9DTQn6sxGA


In [13]:
df_sample100_original_yt = pd.read_csv('../tmp/Sample100.tsv', sep='\t')
df_sample100_original_yt

,Sample_id,whosampled_url,destination_url,source_url
0,S001,https://www.whosampled.com/sample/2238/O.C.-Ti...,https://www.youtube.com/watch?v=6gNmCGQRpcc,https://www.youtube.com/watch?v=DkQFAxSwwjk
1,S002,https://www.whosampled.com/sample/991/Lil-Wayn...,https://www.youtube.com/watch?v=WxPnAErnJ9A,https://www.youtube.com/watch?v=8j8pSu3U7WM
2,S004,https://www.whosampled.com/sample/20324/Common...,https://www.youtube.com/watch?v=_0aP1wjOt7E,https://www.youtube.com/watch?v=3ofnKqgTdSk
3,S005,https://www.whosampled.com/sample/3514/The-Bea...,https://www.youtube.com/watch?v=GX4hoAof2wQ,https://www.youtube.com/watch?v=YqAQiQ63r0A
4,S006,https://www.whosampled.com/sample/1340/Beastie...,https://www.youtube.com/watch?v=Xdutu8sWmbQ,https://www.youtube.com/watch?v=b97hqSDRspw
...,...,...,...,...
94,S131,https://www.whosampled.com/sample/991/Lil-Wayn...,https://www.youtube.com/watch?v=WxPnAErnJ9A,https://www.youtube.com/watch?v=8j8pSu3U7WM
95,S134,https://www.whosampled.com/sample/29796/Nikki-...,https://www.youtube.com/watch?v=VKMZCU6RJE4,https://www.youtube.com/watch?v=lqcX9TPWeNg
96,S135,https://www.whosampled.com/sample/781/Nas-Get-...,https://www.youtube.com/watch?v=IFQfQD2c47g,https://www.youtube.com/watch?v=KRAp5iIL3IA
97,S136,https://www.whosampled.com/sample/781/Nas-Get-...,https://www.youtube.com/watch?v=IFQfQD2c47g,https://www.youtube.com/watch?v=KRAp5iIL3IA


In [17]:
sample100_original_yt_ids = set(
    df_sample100_original_yt[["destination_url", "source_url"]]
    .map(extract_yt_id)
    .stack()
    .dropna()
)
print(len(sample100_original_yt_ids))

136


In [12]:
sample100_yt_ids = set(df_sample100_yt[1].apply(lambda x: x.split('watch?v=')[1]).to_list())
print(len(sample100_yt_ids))

144


In [19]:
print(len(sample100_original_yt_ids.intersection(sample100_yt_ids)))
print(len(sample100_original_yt_ids.difference(sample100_yt_ids)))
print(len(sample100_yt_ids.difference(sample100_original_yt_ids)))

48
88
96


## Build a whosampled dict

In [14]:
track_dict = {}
for _,row in df_sample100_tracks.iterrows():
    track_dict[row.track_id] = {"artist": row.artist, "title": row.title}

In [27]:
whosampled_dict = {}
for _, x in df_sample100_samples.iterrows():
#     whosampled_dict[x.sample_id]
    dct = {
        "original_track_artist": track_dict[x.original_track_id]["artist"],
        "original_track_title": track_dict[x.original_track_id]["title"],
        "sample_track_artist": track_dict[x.sample_track_id]["artist"],
        "sample_track_title": track_dict[x.sample_track_id]["title"],
    }
    orig  = f"{dct['original_track_artist']} - {dct['original_track_title']}"
    samp  = f"{dct['sample_track_artist']} - {dct['sample_track_title']}"
    print(f"{x.sample_id:>3}  |  {samp:<50}  →  {orig}")    

S001  |  OC - Time's Up                                      →  Les Demerle - A Day in the Life
S002  |  Lil Wayne - Dr. Carter                              →  David Axelrod - Holy Thursday
S003  |  Lil Wayne - Dr. Carter                              →  David Axelrod - Holy Thursday
S004  |  Common - In My Own World (Check the Method)         →  Clyde McPhatter - Mixed Up Cup
S005  |  The Beatnuts - Let Off a Couple                     →  Monty Alexander - Love and Happiness
S006  |  Beastie Boys - Rhymin & Stealin'                    →  Led Zeppelin - When The Levee Breaks
S007  |  Blackstreet - No Diggity                            →  Bill Withers - Grandma's Hands
S008  |  Jay-Z - Roc Boys (and the Winner is)                →  Menahan Street Band - Make the Road by Walking
S009  |  A Tribe Called Quest - Can I Kick it?               →  Lou Reed - Walk on the Wild Side
S010  |  House of Pain - Jump Around                         →  Bob & Earl - Harlem Shuffle
S012  |  Will Smith - Mi

In [36]:
sample_whosampled_dict = {}
with open("../tmp/sample100-whosampled.csv") as f:
    reader = csv.reader(f, delimiter=',')
    next(reader)
    for row in reader:
        if row[1]:
            sample_whosampled_dict[row[0]] = row[1]

In [37]:
sample_whosampled_dict

{'S001': '2238',
 'S002': '991',
 'S004': '20324',
 'S005': '3514',
 'S006': '1340',
 'S007': '23',
 'S008': '521',
 'S009': '120',
 'S010': '622',
 'S012': '47',
 'S013': '14037',
 'S014': '1158',
 'S015': '4692',
 'S016': '1164',
 'S017': '6394',
 'S018': '22803',
 'S019': '1764',
 'S020': '1384',
 'S021': '12020',
 'S022': '189',
 'S023': '2064',
 'S024': '4196',
 'S025': '4657',
 'S027': '14255',
 'S028': '14254',
 'S030': '11607',
 'S031': '35245',
 'S032': '11610',
 'S033': '804',
 'S035': '4125',
 'S036': '266',
 'S037': '666',
 'S038': '104',
 'S039': '8327',
 'S040': '153',
 'S041': '3147',
 'S042': '237',
 'S043': '17600',
 'S044': '17182',
 'S046': '249',
 'S047': '10934',
 'S048': '35246',
 'S049': '889',
 'S051': '806',
 'S052': '2291',
 'S054': '12802',
 'S055': '3410',
 'S056': '926',
 'S057': '56949',
 'S060': '6501',
 'S061': '6910',
 'S062': '9640',
 'S063': '2667',
 'S064': '9537',
 'S067': '74883',
 'S068': '13166',
 'S069': '31178',
 'S071': '21298',
 'S072': '2129

## Analyze

In [5]:
sources = set(df_sample100_samples['original_track_id'].str.strip())
receivers = set(df_sample100_samples['sample_track_id'].str.strip())
both = sources & receivers
print(both)

set()


In [8]:
# Rows where the pair appears more than once (keeps all occurrences)
dupes = df_sample100_samples[
    df_sample100_samples.duplicated(subset=['original_track_id', 'sample_track_id'], keep=False)
].sort_values(['original_track_id', 'sample_track_id'])
display(dupes)

,sample_id,original_track_id,sample_track_id,t_original,t_sample,n_repetitions,sample_type,interpolation,comments
1,S002,T003,T004,0,0,4,riff,no,intro
2,S003,T003,T004,34,53,3,riff,no,refrain with horns
101,S131,T003,T004,23,20,9,riff,no,repeated beat (of which 3 included in S2)
7,S008,T014,T013,74,5,10,riff,maybe,beginning of riff counted (rest 'chopped')
100,S130,T014,T013,68,0,1,riff,no,intro not interpolated
11,S013,T023,T022,1,10,51,riff,maybe,piano
99,S129,T023,T022,11,84,2,riff,maybe,saxophone
14,S016,T029,T028,0,37,50,riff,no,
98,S128,T029,T028,5,27,4,riff,no,10 sec refrain
15,S017,T031,T030,8,10,43,riff,no,first half of loop


In [10]:
# Why are there duplicates?
edges = dupes[['original_track_id', 'sample_track_id']].drop_duplicates()
len(edges)

14

## CSV

In [ ]:
import csv
from collections import defaultdict

result = defaultdict(list)

with open(path_csv_samples, newline="", encoding='latin-1') as f:
    reader = csv.DictReader(f)
    for row in reader:
        result[row["sample_track_id"]].append(row["original_track_id"])
result = dict(result)

In [ ]:
len(result)

In [ ]:
max([len(v) for v in result.values()])

In [ ]:
result

In [ ]:
df_tracks = pd.read_csv(path_csv_tracks)
display(df_tracks)

In [ ]:
paths_sample100_audio = list(dir_sample100.rglob('*.mp3'))
print(len(paths_sample100_audio))